# Programación Funcional en Python

## Notebook unificado de clase

En esta sesión vamos a combinar la esencia de varios materiales en una sola ruta de aprendizaje centrada en programación funcional aplicada en Python.

### Objetivos
- Entender qué significa tratar funciones como valores.
- Usar `lambda`, `map()`, `filter()` y `reduce()` con criterio.
- Comparar estas herramientas con list comprehensions, `any()` y `all()`.
- Construir pequeños pipelines funcionales sobre un dataset realista.

### Agenda sugerida (3h)
- 0:00-0:15: introducción y funciones como ciudadanos de primera clase.
- 0:15-0:35: `apply_twice` y paso de funciones.
- 0:35-0:55: `lambda` y transformaciones simples.
- 0:55-1:20: `map`, `filter` y `reduce`.
- 1:20-1:30: recap corto.
- 1:30-1:45: descanso.
- 1:45-2:10: list comprehensions + `any` / `all`.
- 2:10-2:35: `sorted(..., key=...)` y ranking.
- 2:35-2:55: mini-pipeline funcional.
- 2:55-3:00: cierre y reto opcional.


## 1. Introducción breve: funciones como ciudadanos de primera clase

En programación funcional, una función no es solo un bloque de código que se ejecuta cuando la llamas. También puede:

- guardarse en una variable,
- pasarse como argumento,
- devolverse desde otra función.

Esta idea nos permite construir transformaciones más declarativas y reutilizables.

Antes de entrar en lo funcional puro, solo necesitamos recordar una base mínima: una función recibe datos, transforma algo y devuelve un resultado.


In [ ]:
# Functions can be stored in variables and passed to other functions

def add_one(x: int) -> int:
    return x + 1


def apply_to_list(fn, items):
    result = []
    for item in items:
        result.append(fn(item))
    return result


operation = add_one #la funcion se puede guardar en una variable
numbers = [1, 2, 3]

print("Original:", numbers)
print("After applying a function value:", apply_to_list(operation, numbers))#la funcion se puede pasar por parametro a otra funcion


Original: [1, 2, 3]
After applying a function value: [2, 3, 4]


### Ejercicio 1: `apply_twice(fn, x)`

Crea una función `apply_twice` que aplique una función dos veces sobre un valor.

Ejemplo:
- `apply_twice(add_one, 10)` debería devolver `12`.

Todavía no hace falta usar `lambda`: puedes apoyarte en la función `add_one` de la celda anterior.


In [ ]:
# Exercise 1
def add_one(x: int) -> int:
    return x + 1

def apply_twice(fn, number):
    try:
        #Version 1
        """ 
        num = fn(number)
        return fn(num)
        """
        #Version 2
        return fn(fn(number))
    except Exception as e:
        print(f"Error occurred: {e}")
        return None

# Example: debe retornar 8+1+1=10
print(apply_twice(add_one, 8))



10


## 2. `lambda`: funciones pequeñas y expresivas

Las funciones `lambda` son funciones anónimas de una sola expresión.

Son útiles cuando:
- la transformación es corta,
- no merece la pena crear una función con `def`,
- queremos pasar comportamiento rápidamente a `map`, `filter`, `sorted` o funciones propias.

No conviene abusar de ellas si la lógica se vuelve larga o difícil de leer.


In [ ]:
# Lambda functions are useful for short transformations

#Ejemplo 1
lambda x: x ** 2
#Donde: 
    # lambda --	función anónima
    # x	---  parámetro de entrada
    # x ** 2 -- valor de retorno

# Equivale a:
def mifuncion(numero):
    return numero*2
    
# Ejemplo 2: Aqui recibe dos parametros (a y b) 
lambda a, b: a * b 
    
# Equivale a:
def mifuncion(a, b):
    return a*b


In [ ]:


square = lambda x: x ** 2
def calc_square(x):
    return x**2

multiply = lambda a, b: a * b
def calc_multiply(a, b):
    return a*b

words = ["functional", "python", "lambda"]
words_by_length = sorted(words, key=lambda word: len(word))

words_by_length = words[:]  # copia de la lista
for i in range(len(words_by_length)):
    for j in range(len(words_by_length) - 1):
        if len(words_by_length[j]) > len(words_by_length[j + 1]):
            # intercambiar
            words_by_length[j], words_by_length[j + 1] = words_by_length[j + 1], words_by_length[j]

print(words_by_length)

print("Square of 5:", square(5))
print("Multiply 4 * 6:", multiply(4, 6))
print("Sorted by length:", words_by_length)


['python', 'lambda', 'functional']
Square of 5: 25
Multiply 4 * 6: 24
Sorted by length: ['python', 'lambda', 'functional']


### Ejercicio 2: crear una `lambda` útil

Crea una `lambda` llamada `to_fahrenheit` para convertir una temperatura de Celsius a Fahrenheit.

Después, úsala para construir una nueva lista a partir de `celsius`.

Fórmula:
`F = C * (9/5) + 32`

De momento no hace falta usar `map()`: puedes usar un bucle `for` si te resulta más natural.


In [9]:
# Exercise 2
celsius = [0, 10, 20, 30]
print("Celsius:", celsius)

#version1 usando bucle for
farenheit1 = []
def convert_farenheit(temp):
    return (temp * 9/5) + 32

for c in celsius:
    farenheit1.append(convert_farenheit(c))
print("Fahrenheit1:", farenheit1)

#version2 usando lambda
farenheit2 = []
to_fahrenheit = lambda c: (c * 9/5) + 32

for c in celsius:
    farenheit2.append(to_fahrenheit(c))
print("Fahrenheit2:", farenheit2)

# Version3 usando list comprehensive
farenheit3 = [to_fahrenheit(c) for c in celsius]
print("Fahrenheit3:", farenheit3)

# Version4 usando lambda y list comprehensive
farenheit4 = [(c * 9/5) + 32 for c in celsius]
print("Fahrenheit4:", farenheit4)


Celsius: [0, 10, 20, 30]
Fahrenheit1: [32.0, 50.0, 68.0, 86.0]
Fahrenheit2: [32.0, 50.0, 68.0, 86.0]
Fahrenheit3: [32.0, 50.0, 68.0, 86.0]
Fahrenheit4: [32.0, 50.0, 68.0, 86.0]


## 3. `map()`, `filter()` y `reduce()`

Estas tres herramientas son muy representativas del estilo funcional:

- `map(fn, iterable)`: transforma cada elemento.
- `filter(fn, iterable)`: conserva solo los elementos que cumplen una condición.
- `reduce(fn, iterable)`: combina todos los elementos en un único resultado.

`reduce` vive en `functools`, así que hay que importarlo.


In [ ]:
#map(fn, iterable)--- MODIFICA LOS VALORES

fn = lambda x: x*2
iterable = [1, 2, 3,4, 5]

listav1 = list(map(fn, iterable))
listav2 = list(map(lambda x: x*2, iterable))
print(listav2)
# map() aplica una función a cada elemento del iterable.
# Devuelve un nuevo iterable con los elementos transformados.
# Para convertir el resultado en una lista usamos list().

[2, 4, 6, 8, 10]


In [ ]:
#filter(fn, iterable) --- Selecciona solo los valores que cumplen la condicion

fn = lambda x: x > 2
iterable = [1, 2, 3,4, 5]

listav1 = list(filter(fn, iterable))
listav2 = list(filter(lambda x: x > 2, iterable))
print(listav2)
# filter() recorre el iterable y evalúa cada elemento usando la función.
# Solo conserva los elementos para los que la función devuelve True.
# filter() retorna un objeto iterable de tipo filter.
# Para visualizar los resultados como lista, usamos list().

[3, 4, 5]


In [ ]:
# reduce(fn, iterable) -- retorna un unico resultado
# Recorre el iterable acumulando el resultado parcial (acc)
# de cada operación hasta obtener un único valor final.
# La función recibe:
#   acc → acumulador (resultado parcial)
#   x   → elemento actual

# NOTA:
# Cuando NO se proporciona un valor inicial,
# reduce() usa automáticamente el primer elemento
# del iterable como acumulador inicial (acc).

from functools import reduce

fn = lambda acc, x: acc + x
iterable = [1, 2, 3, 4, 5]

resultado = reduce(fn, iterable)
print(resultado)


# Proceso:
# En este ejemplo:
# acc = 1  → primer elemento
# x   = 2  → segundo elemento
# paso 1: acc = 1,  x = 2  → 1 + 2 = 3
# paso 2: acc = 3,  x = 3  → 3 + 3 = 6
# paso 3: acc = 6,  x = 4  → 6 + 4 = 10
# paso 4: acc = 10, x = 5  → 10 + 5 = 15
# Resultado final:
# 15


15


In [ ]:
from functools import reduce

numbers = [1, 2, 3, 4, 5]

# con funcion
def multiplicar_pordos(x):
    return x*2;
mapped1 = []
for x in numbers:
    mapped1.append(multiplicar_pordos(x))
mapped2 = list(map(multiplicar_pordos, numbers))#solo se le pasa la referencia a la funcion, map solo le pasa el parametro

mapped3 = list(map(lambda x: x * 2, numbers))
filtered = list(filter(lambda x: x % 2 == 0, numbers))
reduced = reduce(lambda acc, x: acc + x, numbers)

print("Mapped:", mapped3)
print("Filtered:", filtered)
print("Reduced sum:", reduced)


Mapped: [2, 4, 6, 8, 10]
Filtered: [2, 4]
Reduced sum: 15


## Dataset base: estudiantes

A partir de aquí usaremos un mismo dataset para todos los ejercicios. La idea es aplicar transformaciones funcionales sin cambiar de contexto en cada bloque.


In [ ]:
students = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [5, 6, 4], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": [6, 7, 6], "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
]


def average(grades):
    return sum(grades) / len(grades)

list_averages = []
for a in students:
    list_averages.append((a["name"], average(a["grades"])))

# usando lambda
get_average = lambda student: (student["name"], average(student["grades"]))
for student in students:
    list_averages.append(get_average(student))

for el in list_averages:
    print("Average: ", el)


Average:  ('Ana', 8.0)
Average:  ('Luis', 5.0)
Average:  ('Marta', 9.333333333333334)
Average:  ('Juan', 6.333333333333333)
Average:  ('Sofia', 5.0)


### Ejercicio 3 (`map`): promedios por estudiante

Calcula el promedio de notas por estudiante y genera una lista de tuplas con formato:

`(name, average)`


In [45]:
# Exercise 3
students = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [5, 6, 4], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": [6, 7, 6], "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
]

#Version usando bucle
list_averages = []
for a in students:
    list_averages.append((a["name"], round(average(a["grades"]),2)))
print(list_averages)

#Version usando lambda y map()
map_average = list(map(
    lambda student: (
        (student["name"], round(average(student["grades"]), 2))
    ),
    students 
))
print(map_average)


[('Ana', 8.0), ('Luis', 5.0), ('Marta', 9.33), ('Juan', 6.33), ('Sofia', 5.0)]
[('Ana', 8.0), ('Luis', 5.0), ('Marta', 9.33), ('Juan', 6.33), ('Sofia', 5.0)]


### Ejercicio 4 (`filter`): aprobados con asistencia mínima

Filtra estudiantes que cumplan las dos condiciones:

- promedio `>= 6`
- ausencias `<= 2`


In [20]:
# Exercise 4
estudiantes = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [5, 6, 4], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": [6, 7, 6], "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
]

#VERSION A
# Paso 1: filtrar la lista por los aprobados
def average(estudiante):
    grades = estudiante["grades"]
    return round(sum(grades) / len(grades), 2)

aprobados = list(filter(average, estudiantes))
print("Estudiantes aprobados:")
for s in aprobados:
    print(f"Estudiante: {s["name"]} -- Notas: {s["grades"]}")

# Paso 2: filtrar la lista de aprobados filtro por ausencias
mejores_estudiantes = list(filter(lambda estudiante: estudiante["absences"] <= 2, aprobados))
print("VERSION A -- Mejores estudiantes(aprobados, sin ausencias):")
for s in mejores_estudiantes:
    print(f"Estudiante: {s["name"]} -- Notas: {s["grades"]} -- Ausencias: {s["absences"]}")

#VERSION B : Recorrer la lista de estudiantes e ir guardando en una nueva el que cumpla las dos condiciones
best_students = []

for student in estudiantes:
    grades = student["grades"]
    absences = student["absences"]
    average = round(sum(grades) / len(grades), 2)
    if average >= 6 and absences <= 2:
        best_students.append(({"estudiante":student["name"]}, 
                               {"Average":average}, 
                               {"Ausencias": student["absences"]}
                               ))

print("VERSION B -- Mejores estudiantes(aprobados, sin ausencias):")
for s in best_students:
    print(s)

#VERSION C
def find_best_student(student):
    grades = student["grades"]
    absences = student["absences"]
    average = round(sum(grades) / len(grades), 2)
    return average >= 6 and absences <= 2

best_students = list(filter(find_best_student, estudiantes))
print("VERSION C -- Mejores estudiantes(aprobados, sin ausencias):")
for s in best_students:
    print(f"Estudiante: {s["name"]} -- Notas: {s["grades"]} -- Ausencias: {s["absences"]}")





Estudiantes aprobados:
Estudiante: Ana -- Notas: [8, 7, 9]
Estudiante: Luis -- Notas: [5, 6, 4]
Estudiante: Marta -- Notas: [9, 9, 10]
Estudiante: Juan -- Notas: [6, 7, 6]
Estudiante: Sofia -- Notas: [4, 5, 6]
VERSION A -- Mejores estudiantes(aprobados, sin ausencias):
Estudiante: Ana -- Notas: [8, 7, 9] -- Ausencias: 1
Estudiante: Marta -- Notas: [9, 9, 10] -- Ausencias: 0
Estudiante: Juan -- Notas: [6, 7, 6] -- Ausencias: 2
VERSION B -- Mejores estudiantes(aprobados, sin ausencias):
({'estudiante': 'Ana'}, {'Average': 8.0}, {'Ausencias': 1})
({'estudiante': 'Marta'}, {'Average': 9.33}, {'Ausencias': 0})
({'estudiante': 'Juan'}, {'Average': 6.33}, {'Ausencias': 2})
VERSION C -- Mejores estudiantes(aprobados, sin ausencias):
Estudiante: Ana -- Notas: [8, 7, 9] -- Ausencias: 1
Estudiante: Marta -- Notas: [9, 9, 10] -- Ausencias: 0
Estudiante: Juan -- Notas: [6, 7, 6] -- Ausencias: 2


### Ejercicio 5 (`reduce`): suma total de promedios

Usa `reduce` para sumar los promedios de todos los estudiantes.


In [ ]:
# Exercise 5
from functools import reduce

students = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [5, 6, 4], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": [6, 7, 6], "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
]

def calculate_average(grades):
    return sum(grades) / len(grades)

#Version usando bucle
suma_total = 0
for student in students:
    average = calculate_average(student["grades"])
    suma_total += average
print("Average total:", suma_total)


#Version usando funcion lambda y reduce(fn, iterable)
# reduce sirve para acumular valores recorriendo una lista.
# acc → acumulador (lo que llevas sumado hasta ahora)
# student → elemento actual de la lista
total_average_sum = reduce(
    lambda acc, student: 
        acc + calculate_average(student["grades"]),
          students,
          0
    )
print(total_average_sum)

#ME QUEDE AQUI TODO



Average total: 33.66666666666667
33.66666666666667


## Pausa sugerida (15 min)

Este es un buen momento para parar, comentar dudas y volver con energía a la segunda mitad de la sesión.


## 4. Alternativas prácticas: list comprehensions + `any()` / `all()`

En Python real, muchas veces `map` y `filter` se sustituyen por comprensiones de listas porque resultan más legibles.

Además:
- `any(iterable)` devuelve `True` si al menos un elemento cumple la condición.
- `all(iterable)` devuelve `True` si todos los elementos cumplen la condición.


In [ ]:
numbers = [1, 2, 3, 4, 5]

squares_map = list(map(lambda x: x ** 2, numbers))
squares_comp = [x ** 2 for x in numbers]

has_many_absences = any(student["absences"] > 3 for student in students)# primero se pone la condicion y a continuacion el iterable
all_passing = all(average(student["grades"]) >= 6 for student in students)

print("Map version:", squares_map)
print("Comprehension version:", squares_comp)
print("Any student with many absences:", has_many_absences)
print("All students passing:", all_passing)


### Ejercicio 6: reescribe con list comprehension

Genera una lista con los nombres de los estudiantes que tienen `0` ausencias.


In [ ]:
# Exercise 6
students = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [5, 6, 4], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": [6, 7, 6], "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
]
#primera version sin list comprehension
perfect_attendance = []
for estudiante in students:
    if estudiante["absences"] == 0:
        perfect_attendance.append(estudiante["name"])

print(perfect_attendance)

#primera version COn list comprehension

perfect_attendance = [estudiante["name"] for estudiante in students if estudiante["absences"] == 0]
print(perfect_attendance)


['Marta']
['Marta']


### Ejercicio 7: alertas con `any()` y `all()`

1. Usa `any()` para comprobar si algún estudiante tiene **alguna** nota menor que `5`.
2. Usa `all()` para comprobar si todos los estudiantes tienen promedio `>= 6`.


In [ ]:
# Exercise 7
students = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [5, 6, 4], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": [6, 7, 6], "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
]
#1.Usa `any()` para comprobar si algún estudiante tiene **alguna** nota menor que `5`.

# Version usando bucle for
has_many_absences1 = False
for estudiante in students:
    for nota in estudiante["grades"]:
        if nota < 5:
            has_many_absences1 = True
print("has_many_absences1: ",has_many_absences1)


# Version usando any() y list comprehensive
has_many_absences2 = any(
    grade < 5     
    for student in students
    for grade in student["grades"]
)
print("has_many_absences2: ",has_many_absences2)



has_many_absences1:  True
has_many_absences2:  True
has_low_grade1:  False
has_low_grade2:  False


In [ ]:
# Ejercicio 2. Usa `all()` para comprobar si todos los estudiantes tienen promedio `>= 6`.

students = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [5, 6, 4], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": [6, 7, 6], "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
]

def average(grades):
    return sum(grades) / len(grades)


# Version usando bucle for
has_low_grade1 = False
for student in students:
    if average(student["grades"]) >= 6:
        has_low_grade = True 
print("has_low_grade1: ",has_low_grade1)

# Version usando all() y list comprehensive
has_low_grade2 = all(
    average(student["grades"]) >= 6
     for student in students
)
print("has_low_grade2: ",has_low_grade2)

## 5. `sorted(..., key=...)` y ranking

`sorted()` permite ordenar una colección sin modificar la original.

Cuando usamos `key=...`, podemos decidir por qué criterio ordenar. Si devolvemos una tupla en `key`, podemos combinar varios criterios a la vez.

En este caso ordenaremos por:
1. promedio descendente,
2. ausencias ascendentes.


In [ ]:
students = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [5, 6, 4], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": [6, 7, 6], "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
]

ranked_students = sorted(
    students,
    key=lambda student: (-average(student["grades"]), student["absences"]),
)

for student in ranked_students:
    print(f"{student['name']}: Average = {average(student['grades']):.2f}, Absences = {student['absences']}")

### Ejercicio 8: top 3

Genera una lista con los 3 mejores estudiantes usando el mismo criterio de ordenación:

- mejor promedio primero,
- en caso de empate, menos ausencias.


In [ ]:
# Exercise 8
ranked_students = sorted(
students,
key=lambda student: (-average(student["grades"]), student["absences"]),
)

if len(ranked_students) > 3:
    top_3 = ranked_students[:3]
else:
    top_3 = ranked_students

for student in top_3:
    print(f"{student['name']}: Average = {average(student['grades']):.2f}, Absences = {student['absences']}")

## 6. Mini-pipeline funcional

Un pipeline funcional combina varios pasos pequeños y claros:

1. limpiar datos,
2. transformar,
3. filtrar,
4. ordenar,
5. devolver un resultado final.

La idea no es hacer “magia” en una sola línea, sino encadenar pasos que expresen bien la intención.


### Ejercicio 9: pipeline completo

A partir de un dataset con entradas inválidas:

1. Elimina estudiantes sin notas válidas.
2. Calcula el promedio y crea un nuevo diccionario con `name`, `average` y `absences`.
3. Filtra estudiantes con promedio `>= 6` y ausencias `<= 2`.
4. Ordena por promedio descendente y ausencias ascendente.


In [ ]:
# Exercise 9
raw_students = [
    {"name": "Ana", "grades": [8, 7, 9], "absences": 1},
    {"name": "Luis", "grades": [], "absences": 3},
    {"name": "Marta", "grades": [9, 9, 10], "absences": 0},
    {"name": "Juan", "grades": None, "absences": 2},
    {"name": "Sofia", "grades": [4, 5, 6], "absences": 4},
    {"name": "Elena", "grades": [10, 9, 9], "absences": 0},
]

# result = ...
# print(result)


### Reto opcional: lista de honor

Si terminas antes, crea un pipeline que devuelva solo los nombres de estudiantes que cumplan:

- promedio `>= 8.5`
- ausencias `== 0`

Intenta hacerlo con pasos pequeños y legibles.


In [ ]:
# Optional challenge

# honor_roll = ...
# print(honor_roll)


## Cierre rápido

Ideas clave de la sesión:

- una función puede comportarse como un valor,
- `lambda`, `map`, `filter` y `reduce` son herramientas útiles, pero no siempre las más legibles,
- list comprehensions, `any`, `all` y `sorted` suelen aparecer mucho en código real,
- la programación funcional brilla especialmente cuando trabajamos con colecciones y transformaciones encadenadas.
